# ContraTICO — NER Extension Pipeline

Full entity-aware pipeline for ContraTICO:
1. **NER Extraction** — Extract biomedical entities (`ner_extraction_contratico.py`)
2. **QG** — Entity-aware question generation (`qg_entity_aware_contratico.py`)
3. **QA Source** — Answer on English source (`qa_entity_contratico.py`)
4. **QA BT** — Answer on perturbed BT × 5 languages (`qa_entity_contratico.py`)
5. **Evaluation** — String comparison and SBERT

| Parameter | Values |
|-----------|--------|
| NER model | d4data/biomedical-ner-all |
| LLM | Qwen/Qwen2.5-3B-Instruct |
| Languages | es, fr, hi, tl, zh |
| Perturbations | alteration, omission |

## 0. Environment Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
else:
    PROJECT_ROOT = os.getcwd()

if not os.path.exists(PROJECT_ROOT) and (IN_KAGGLE or IN_COLAB):
    subprocess.run(['git', 'clone',
                    'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                    PROJECT_ROOT], check=True)

print(f'Project root: {PROJECT_ROOT}')

## 1. Configuration

In [ ]:
NER_DIR       = f'{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/ner-extension'
CODE_DIR      = f'{NER_DIR}/code'
DATA_DIR      = f'{PROJECT_ROOT}/contraTICO'

LANGUAGES     = ['es', 'fr', 'hi', 'tl', 'zh']
PERTURBATIONS = ['alteration', 'omission']

# Create output directories
for d in ['results/ner', 'results/qg', 'results/qa/source',
          'results/qa/bt', 'results/sbert', 'results/string_comparison']:
    os.makedirs(f'{NER_DIR}/{d}', exist_ok=True)

print(f'NER dir: {NER_DIR}')
print(f'Code dir: {CODE_DIR}')

## 2. NER Extraction

Extracts biomedical entities from ContraTICO source sentences.

In [ ]:
ner_output = f'{NER_DIR}/results/ner/ner_output.jsonl'

# ContraTICO uses one representative input file
input_file = f'{DATA_DIR}/en-es/alteration.jsonl'

cmd = [
    sys.executable, '-u', f'{CODE_DIR}/ner_extraction_contratico.py',
    '--input_path', input_file,
    '--output_path', ner_output
]

print('Running NER extraction...')
subprocess.run(cmd, check=True)
print(f'✓ NER extraction → {ner_output}')

## 3. QG — Entity-Aware Question Generation

Generates questions specific to each extracted entity using `qg_entity_aware_contratico.py`.

In [ ]:
qg_output = f'{NER_DIR}/results/qg/qg_entity_aware.jsonl'

cmd = [
    sys.executable, '-u', f'{CODE_DIR}/qg_entity_aware_contratico.py',
    '--input_path', ner_output,
    '--output_path', qg_output
]

print('Running entity-aware QG...')
subprocess.run(cmd, check=True)
print(f'✓ QG → {qg_output}')

## 4. QA Source — Entity-Aware Answers on Source Sentences

In [ ]:
qa_src_output = f'{NER_DIR}/results/qa/source/qa_source.jsonl'

cmd = [
    sys.executable, '-u', f'{CODE_DIR}/qa_entity_contratico.py',
    '--qg_path', qg_output,
    '--qa_source_path', qa_src_output,
    '--contratico_path', f'{DATA_DIR}/en-es/alteration.jsonl',
    '--lang', 'es',
    '--output_path', qa_src_output
]

print('Running QA Source...')
subprocess.run(cmd, check=True)
print(f'✓ QA Source → {qa_src_output}')

## 5. QA BT — Entity-Aware Answers on Perturbed Text (all languages × perturbations)

In [ ]:
for lang in LANGUAGES:
    for perturbation in PERTURBATIONS:
        qa_bt_output = f'{NER_DIR}/results/qa/bt/qa_{lang}_{perturbation}.jsonl'
        contratico_file = f'{DATA_DIR}/en-{lang}/{perturbation}.jsonl'

        if not os.path.exists(contratico_file):
            print(f'  ⚠ {contratico_file} not found, skipping')
            continue

        cmd = [
            sys.executable, '-u', f'{CODE_DIR}/qa_entity_contratico.py',
            '--qg_path', qg_output,
            '--qa_source_path', qa_src_output,
            '--contratico_path', contratico_file,
            '--lang', lang,
            '--output_path', qa_bt_output
        ]

        print(f'Running QA BT [{lang}/{perturbation}]...')
        subprocess.run(cmd, check=True)
        print(f'✓ QA BT [{lang}/{perturbation}] → {qa_bt_output}')

print('\n✓ All QA complete!')

## 6. String Comparison

In [ ]:
cmd = [
    sys.executable, '-u', f'{CODE_DIR}/string_comparison_contratico.py',
    '--base_dir', NER_DIR
]
print('Running string comparison...')
subprocess.run(cmd, check=True)
print('✓ String comparison complete!')

## 7. SBERT Evaluation

In [ ]:
cmd = [
    sys.executable, '-u', f'{CODE_DIR}/sbert_contratico.py',
    '--base_dir', NER_DIR
]
print('Running SBERT evaluation...')
subprocess.run(cmd, check=True)
print('✓ SBERT evaluation complete!')

## Summary

NER pipeline complete:
```
ner-extension/results/
├── ner/ner_output.jsonl
├── qg/qg_entity_aware.jsonl
├── qa/source/qa_source.jsonl
├── qa/bt/qa_{lang}_{perturbation}.jsonl
├── sbert/
└── string_comparison/
```